In [1]:
import dimod

# Optimization problems

Optimization problems are a central topic in computer science, mathematics and quantum computing. In general, they consist of finding the best possible solution among many alternatives, according to a given objective function. This objective can represent a cost to minimize, a profit to maximize, or any quantity that measures how good a solution is.

In this notebook, we focus on binary optimization problems, where the decision variables can only take the values 0 or 1. These problems are especially relevant because many real situations, such as selecting items, assigning resources or finding optimal configurations, can be expressed in this form. To formulate them, we will use the QUBO framework and its relation with the Ising model.

# Quadratic Unconstrained Binary Optimization 

The Quadratic Unconstrained Binary Optimization (QUBO) framework helps us formulate optimization problems using binary variables $x_i\in\{0,1\}$. The cost function to minimize is given by:

$$
f(x) = \sum_{i<j}Q_{i,j}x_ix_j + \sum_i Q_{i,i}x_i = x^TQx
$$

where $Q_{i,j}$ is the strength between the variable $i$ and $j$.

The term "unconstrained" does not mean that this formulation cannot be used for problems with restrictions. The procedure is to include the constraints inside the objective function by adding penalty terms. These penalties increase the value of the function when a constraint is violated, so the minimization process naturally favors solutions that satisfy the original restrictions.

In general, an optimization problem can be translated into QUBO by following three steps. First, the decision variables are encoded as binary variables. Then, the objective function is written in terms of these variables. Finally, each constraint is converted into a quadratic penalty term and added to the objective function. Therefore, the final QUBO function contains both the original goal of the problem and the penalties that enforce valid solutions:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda f_{penalty}(x)
$$

where $\lambda$ is a positive parameter that controls the importance of satisfying the constraints. The value of $\lambda$ must be chosen carefully. If it is too small, the optimizer may prefer solutions that improve the original objective function but violate the constraints. On the other hand, if it is too large, the penalty terms can dominate the optimization and make the objective function less relevant. In practice, $\lambda$ is usually selected large enough to ensure that any constraint violation is more costly than the possible improvement obtained in the original objective function.

Let's first consider linear constraints of the form:

$$
Ax= b
$$

It's very easy to deal with this constraints since we can directly add them to the cost function as:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda \|Ax-b\|^2
$$

Now, let's consider inequality constraints of the form:

$$
Ax \leq b
$$

The method to translate inequality constraints into equality constraints is to add slack variables. These variables represent the difference between the right-hand side and the left-hand side of the inequality.

For example, let's consider the constraint:

$$
3x_0 - x_1 + 3x_2 \leq 5
$$

First, we need to find the minimum value that the left-hand side can take. To do this, we set the variables with positive coefficients to 0 and the variables with negative coefficients to 1. In this case, the minimum value is:

$$
3\cdot 0 - 1 + 3\cdot 0 = -1
$$

Therefore, the slack variable must be able to represent values from 0 to 6, since:

$$
5 - (-1) = 6
$$

Since 3 bits are enough to represent values up to 6, we introduce three binary slack variables $y_0$, $y_1$ and $y_2$:

$$
s = y_0 + 2y_1 + 4y_2
$$

Then, the inequality can be rewritten as the following equality:

$$
3x_0 - x_1 + 3x_2 + y_0 + 2y_1 + 4y_2 = 5
$$

Finally, we have an equality constraint, so we can add it to the QUBO objective as a penalty term:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda (3x_0 - x_1 + 3x_2 + y_0 + 2y_1 + 4y_2 - 5)^2
$$



# Ising Model


The Ising model is a mathematical model originally introduced to describe magnetic systems, especially the interaction between particles with spin. These spins are usually arranged in a lattice, and each spin can point in one of two possible directions. The energy of the system depends on the interaction between neighboring spins and on the effect of external local magnetic fields.

The Hamiltonian, which represents the energy of the system, is given by:

$$
H(s) = \sum_{i<j} J_{i,j}s_is_j+\sum_i h_is_i
$$

where $s_i \in \{-1,1\}$ represents the spin variables, $J_{i,j}$ is the strength of the interaction between spins $i$ and $j$, and $h_i$ is the local magnetic field acting on spin $i$. 

This formulation is of great interest because many optimization problems can be expressed as particular cases of the Ising model. Moreover, since the spin variables take values $s_i \in \{-1,1\}$, the formulation can be naturally translated to a quantum computer, where the eigenvalues of the Pauli-$Z$ operator also take these values.

The Ising model is closely related to the QUBO formulation. The main difference is that QUBO uses binary variables $x_i \in \{0,1\}$, while the Ising model uses spin variables $s_i \in \{-1,1\}$. Both formulations can be transformed into each other using the change of variables:

$$
s_i = 2x_i - 1
$$

or equivalently,

$$
x_i = \frac{s_i+1}{2}
$$

Therefore, a QUBO problem can be rewritten as an Ising Hamiltonian by substituting this relation into the objective function and grouping the resulting terms. Both formulations represent the same optimization problem, up to an additive constant that does not affect the minimum.

# Quantum annealing